# Week 10 Lab — Learning a Traffic Controller from Logs
**Imitation Learning (BC) + Support-Aware Offline Q-Learning**

**Teaching simulation only.** This is not a physical traffic-light controller. Train exclusively on logged episodes; evaluation in a held-out simulator is for pedagogy. The second algorithm is a simplified, tabular *support-aware* method, **not CQL**.

**Questions:** Can we clone a fixed heuristic? Can offline reward data improve it? What changes with data coverage?

In [ ]:
%pip -q install numpy pandas matplotlib torch

## 1. A simple traffic model
Two directions compete for green. Action 0 means *keep*; 1 means *request a phase switch*. The simulation enforces a minimum green duration. Queues, phase, and elapsed green time are observations. Congestion cost is total queue length. At each step arrivals are Poisson; departures are bounded by service capacity.

In [ ]:
import numpy as np, pandas as pd, torch
from torch import nn
from matplotlib import pyplot as plt

torch.manual_seed(7)

class TrafficEnv:
    def __init__(self, horizon=80, arrival_rates=(2.5, 2.0), capacity=5, min_green=3):
        self.horizon, self.rates = horizon, np.array(arrival_rates)
        self.capacity, self.min_green = capacity, min_green
    def reset(self, seed=0):
        self.rng = np.random.default_rng(seed); self.queues=np.zeros(2,dtype=float)
        self.phase=0; self.elapsed=0; self.t=0
        return self.obs()
    def obs(self):
        return np.array([self.queues[0], self.queues[1], self.phase, min(self.elapsed,12)],dtype=np.float32)
    def step(self, action):
        switched = bool(action==1 and self.elapsed>=self.min_green)
        if switched: self.phase=1-self.phase; self.elapsed=0
        self.queues += self.rng.poisson(self.rates)
        departing = min(self.capacity,self.queues[self.phase])
        self.queues[self.phase] -= departing
        reward = -float(self.queues.sum()) - 0.4*switched
        self.elapsed+=1; self.t+=1
        done=self.t>=self.horizon
        return self.obs(),reward,done,{'queue':float(self.queues.sum()), 'switch':int(switched)}

def heuristic(obs):
    q0,q1,p,elapsed=obs
    # imperfect controller: threshold to avoid switching too often
    other = q1 if p==0 else q0
    current = q0 if p==0 else q1
    return int(elapsed>=3 and other > current + 3)

def random_policy(obs): return int(np.random.random()<.25)

def rollout(policy, seed=0, rates=(2.5,2.0), collect=False):
    env=TrafficEnv(arrival_rates=rates); s=env.reset(seed); data=[]; metrics=[]
    for step in range(env.horizon):
        a=int(policy(s)); nxt,r,done,info=env.step(a)
        data.append((s.copy(),a,r,nxt.copy(),done))
        metrics.append(info); s=nxt
        if done: break
    summary={'return':sum(t[2] for t in data),
             'mean_queue':np.mean([x['queue'] for x in metrics]),
             'max_queue':max(x['queue'] for x in metrics),
             'switches':sum(x['switch'] for x in metrics)}
    return (data,summary) if collect else summary

print('Sample expert episode:', rollout(heuristic, seed=1))

## 2. Log trajectories, then freeze the dataset
The expert is deliberately imperfect. We include 15% random behavior to support more than one action in some states. Once collected, learners cannot call the simulation `step()` until evaluation.

In [ ]:
rng=np.random.default_rng(34)
def behavioral_policy(obs):
    return int(rng.integers(0,2)) if rng.random()<.15 else heuristic(obs)

logs=[]
for episode in range(220):
    traj,_=rollout(behavioral_policy,seed=episode,collect=True)
    logs.extend(traj)
S=np.stack([t[0] for t in logs]); A=np.array([t[1] for t in logs]);
R=np.array([t[2] for t in logs]); NS=np.stack([t[3] for t in logs]);
DONE=np.array([t[4] for t in logs]);
print('Logged transitions:',len(logs),'action counts:',np.bincount(A))
print(pd.DataFrame({'q_NS':S[:8,0],'q_EW':S[:8,1], 'phase':S[:8,2],
                    'action':A[:8], 'reward':R[:8]}))

## 3. Train behavior cloning
This is ordinary supervised learning: predict the logged action from the logged state. The demonstration policy is a mix of heuristic and exploratory actions, so 100% action accuracy is neither expected nor desirable as a deployment metric.

In [ ]:
scale=np.array([30.,30.,1.,12.],dtype=np.float32)
X=torch.tensor(np.clip(S/scale,0,5),dtype=torch.float32)
y=torch.tensor(A,dtype=torch.long)
net=nn.Sequential(nn.Linear(4,64),nn.ReLU(),nn.Linear(64,64),nn.ReLU(),nn.Linear(64,2))
opt=torch.optim.Adam(net.parameters(),lr=.003)
for epoch in range(80):
    indices=torch.randperm(len(X))
    for ix in indices.split(512):
        loss=nn.functional.cross_entropy(net(X[ix]),y[ix]); opt.zero_grad();loss.backward();opt.step()
net.eval()
def bc_policy(obs):
    with torch.no_grad():
        x=torch.tensor(np.clip(obs/scale,0,5),dtype=torch.float32)
        return int(net(x).argmax())
with torch.no_grad(): accuracy=(net(X).argmax(1)==y).float().mean().item()
print('BC training-set action agreement:',round(accuracy,3))

## 4. Offline reward learning using tabular state aggregation
For demonstration purposes we discretize states and use **support-aware fitted Q updates**. During optimization we never sample the simulator. We use a minimum count for logged `(state, action)` pairs and penalize unsupported actions by excluding them from greedy maximization. This is a simplistic approximation that can itself fail: it does not establish statistical confidence or genuine policy improvement.

In [ ]:
def discretize(x):
    # broad bins: queue lengths, phase, and elapsed-green indicator
    return (int(min(x[0]//6,7)),int(min(x[1]//6,7)),int(x[2]),int(x[3]>=3))
keys=[discretize(s) for s in S]; nextkeys=[discretize(s) for s in NS]
from collections import Counter
counts=Counter(zip(keys,A))
Q={}
min_count=3

def supported_actions(key):
    return [a for a in (0,1) if counts[(key,a)]>=min_count]
def max_supported_value(key):
    aa=supported_actions(key)
    return max([Q.get((key,a),0.) for a in aa], default=0.)

gamma=.96
for iteration in range(90):
    totals={}; nums={}
    for key,a,r,nkey,done in zip(keys,A,R,nextkeys,DONE):
        if counts[(key,int(a))]<min_count: continue
        target=float(r) + (0 if done else gamma*max_supported_value(nkey))
        pair=(key,int(a)); totals[pair]=totals.get(pair,0.)+target
        nums[pair]=nums.get(pair,0)+1
    newQ={pair:totals[pair]/nums[pair] for pair in totals}
    Q=newQ

def offline_policy(obs):
    k=discretize(obs); aa=supported_actions(k)
    if not aa: return heuristic(obs)  # explicitly declared fallback outside dataset support
    return max(aa,key=lambda a:Q.get((k,a),-1e9))
print('Supported state/action pairs:',len(Q))

## 5. Evaluate on held-out simulation seeds
The training algorithms used only logged transitions. Evaluation opens the simulator *after* training, using identical exogenous arrival seeds for each controller (not necessarily identical state trajectories). These evaluations are **not offline policy evaluation**.

In [ ]:
policies={'Heuristic':heuristic,'Behavior cloning':bc_policy,'Offline support-aware Q':offline_policy}
rows=[]
for name,policy in policies.items():
    for seed in range(1000,1050):
        rows.append({'policy':name,'seed':seed,**rollout(policy,seed=seed)})
results=pd.DataFrame(rows)
print(results.groupby('policy')[['return','mean_queue','max_queue','switches']].agg(['mean','std']).round(2))
fig,ax=plt.subplots(figsize=(8,4))
results.groupby('policy')['mean_queue'].mean().sort_values().plot.bar(ax=ax)
ax.set_ylabel('Mean vehicles waiting (lower is better)');ax.set_title('Held-out simulator evaluation')
plt.tight_layout();plt.show()

## 6. Shift the traffic distribution
We now test an arrival pattern unlike the logged training environment. Interpretation: a good held-out score in the original setting does not guarantee robust performance.

In [ ]:
shift=[]
for name,policy in policies.items():
    for seed in range(2000,2050):
        shift.append({'policy':name,**rollout(policy,seed=seed,rates=(3.5,1.5))})
print(pd.DataFrame(shift).groupby('policy')[['return','mean_queue','max_queue']].agg(['mean','std']).round(2))

## 7. Investigations and reflection
1. Change exploration from 15% to 0%, 5%, and 35%. How does offline data coverage affect performance?
2. Compare action agreement with held-out episode return. Why can they disagree?
3. Change `min_count` (1, 3, 10). When do the support assumptions become too restrictive?
4. Compare training and shifted arrivals; explain any performance degradation.
5. Which of these policies could you legitimately certify for a real traffic intersection? **None**, based on this simulation alone.
6. For graduate students: explain why taking a max over learned Q values for unseen actions can amplify value errors, and how CQL addresses this differently.

**Instructor note:** The naive offline method may underperform BC or the heuristic. This is an intended lesson about offline learning and data coverage, not a bug. The toy simulator omits pedestrians, yellow/red clearance, coordination with adjacent signals, sensors and measurement noise.